In [0]:
import time
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
%run ../00_utils/common

In [0]:
BATCH = batch_id()
TODAY = F.current_date()

## fact_ventas
Base de todas las agregaciones. Enriquecida con dims.
Linaje:
- vr_venta_neto <- qty_vendida * precio_unitario_venta - descuento_aplicado
- flag_descuento <- 1 si descuento_aplicado > 0

In [0]:
def build_fact_ventas():
    print(" === Inicio fact_ventas ===")
    t0 = time.time()
    ventas = spark.table(f"{CATALOG}.silver.trans_ventas")
    tiendas = spark.table(f"{CATALOG}.gold.dim_tiendas").select("id_tienda", "id_pais", "id_ciudad", "tipo_tienda", "zona_distribucion")
    arts = spark.table(f"{CATALOG}.gold.dim_productos").select("art_id", "id_categ_n1", "id_categ_n2", "id_proveedor")
    clientes = spark.table(f"{CATALOG}.gold.dim_clientes").select("id_miembro")

    df = (ventas
          .withColumn("vr_venta_neto",F.round(F.col("qty_vendida") * F.col("precio_unitario_venta") - F.col("descuento_aplicado"), 2))
          .withColumn("flag_descuento", F.when(F.col("descuento_aplicado") > 0, F.lit(1)).otherwise(F.lit(0)))
          .withColumn("id_miembro",
              F.when(F.col("id_miembro") == "ANONIMO", F.lit("ANONIMO"))
               .when(F.col("id_miembro").isNull(), F.lit("ANONIMO"))
               .otherwise(F.col("id_miembro")))
          .join(tiendas, on="id_tienda", how="left")
          .join(arts, on="art_id", how="left")
          # Validar id_miembro contra dim — si no existe -> ANONIMO
          .join(clientes.withColumnRenamed("id_miembro", "_mbr_valid"), F.col("id_miembro") == F.col("_mbr_valid"), "left")
          .withColumn("id_miembro",
              F.when(F.col("_mbr_valid").isNull(), F.lit("ANONIMO"))
               .otherwise(F.col("id_miembro")))
          .drop("_mbr_valid", "_ingested_at", "_source_system", "_batch_id")
          .withColumn("_batch_id", F.lit(BATCH)))

    (df.write.format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .partitionBy("id_pais", "canal_venta")
       .saveAsTable(f"{CATALOG}.gold.fact_ventas"))
    log_run("gold", "fact_ventas", df.count(), 0, time.time() - t0, batch=BATCH)
    print(f"  [DONE] gold.fact_ventas")


## fact_inventario
Necesidad: identificar quiebre en próximos 7 días considerando stock actual, consumo 14 días y tiempo de reabastecimiento. Linaje:
- promedio_consumo_14d <- avg(qty_vendida) últimos 14d por art+tienda
- cobertura_dias <-  stock_fisico / promedio_consumo_14d
- dias_hasta_quiebre <- cobertura_dias (cuántos días dura el stock)
- tiempo_repo_dias <- heredado de dim_productos (via proveedor)
- alerta_quiebre <- cobertura_dias < 7 AND promedio_consumo_14d > 0
- requiere_reposicion <- cobertura_dias <= tiempo_repo_dias (llegará sin stock)
- dif_stock_minimo <- stock_fisico - stock_minimo_config

In [0]:
def build_fact_inventario():
    print(" === Inicio fact_inventario ===")
    t0 = time.time()
    stock = spark.table(f"{CATALOG}.silver.inv_stock_diario")
    ventas = spark.table(f"{CATALOG}.silver.trans_ventas")
    arts = spark.table(f"{CATALOG}.gold.dim_productos").select("art_id", "id_categ_n1", "tiempo_repo_dias")

    consumo_14d = (ventas
                   .filter(F.col("fec_trans") >= F.date_sub(TODAY, 14))
                   .groupBy("art_id", "id_tienda")
                   .agg(F.avg("qty_vendida").alias("promedio_consumo_14d")))

    df = (stock
          .join(consumo_14d, on=["art_id", "id_tienda"], how="left")
          .join(arts, on="art_id", how="left")
          .withColumn("promedio_consumo_14d",
              F.coalesce(F.col("promedio_consumo_14d"), F.lit(0.0)))
          .withColumn("cobertura_dias",
              F.when(F.col("promedio_consumo_14d") > 0,
                     F.round(F.col("stock_fisico") / F.col("promedio_consumo_14d"), 1))
               .otherwise(F.lit(None)))
          # Alerta: stock no alcanza 7 días y hay consumo real
          .withColumn("alerta_quiebre",
              F.when(
                  (F.col("cobertura_dias") < 7) & (F.col("promedio_consumo_14d") > 0),
                  F.lit(1)).otherwise(F.lit(0)))
          # Reposición urgente: el stock se agota antes de que llegue el reabastecimiento
          .withColumn("requiere_reposicion",
              F.when(
                  F.col("cobertura_dias").isNotNull() &
                  (F.col("cobertura_dias") <= F.col("tiempo_repo_dias")),
                  F.lit(1)).otherwise(F.lit(0)))
          .withColumn("dif_stock_minimo", F.col("stock_fisico") - F.col("stock_minimo_config"))
          .drop("_ingested_at", "_source_system", "_batch_id")
          .withColumn("_batch_id", F.lit(BATCH)))

    (df.write.format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .partitionBy("fec_snapshot")
       .saveAsTable(f"{CATALOG}.gold.fact_inventario"))
    log_run("gold", "fact_inventario", df.count(), 0, time.time() - t0, batch=BATCH)
    print(f"  [DONE] gold.fact_inventario")


## fact_devoluciones
Necesidad: analizar por motivo, categoría, proveedor y canal. Linaje:
- precio_original <-  precio_unitario_venta de la transacción origen
- vr_devolucion_total <- qty_devuelta * precio_original
- tasa_devolucion_pct <- qty_devuelta / qty_vendida_cat * 100
  - por categoría nivel 1 y canal de venta

In [0]:
def build_fact_devoluciones():
    print(" === Inicio fact_devoluciones ===")
    t0 = time.time()
    devs = spark.table(f"{CATALOG}.silver.post_devoluciones")
    ventas = spark.table(f"{CATALOG}.silver.trans_ventas").select("id_trans", "precio_unitario_venta", "qty_vendida", "canal_venta")
    arts = spark.table(f"{CATALOG}.gold.dim_productos").select("art_id", "id_categ_n1", "id_proveedor")

    # Unidades vendidas por categoría y canal — denominador de tasa
    ventas_cat = (spark.table(f"{CATALOG}.silver.trans_ventas")
                  .join(arts.select("art_id", "id_categ_n1"), on="art_id", how="left")
                  .groupBy("id_categ_n1", "canal_venta")
                  .agg(F.sum("qty_vendida").alias("qty_vendida_cat")))

    df = (devs
          .join(ventas, devs.id_trans_origen == ventas.id_trans, how="left")
          .join(arts,   on="art_id", how="left")
          .join(ventas_cat, on=["id_categ_n1", "canal_venta"], how="left")
          .withColumn("precio_original", F.coalesce(F.col("precio_unitario_venta"), F.lit(0.0)))
          .withColumn("vr_devolucion_total", F.round(F.col("qty_devuelta") * F.col("precio_original"), 2))
          .withColumn("tasa_devolucion_pct",
              F.when(F.col("qty_vendida_cat") > 0,
                     F.round(F.col("qty_devuelta") / F.col("qty_vendida_cat") * 100, 4))
               .otherwise(F.lit(None)))
          .drop("id_trans", "precio_unitario_venta", "qty_vendida", "qty_vendida_cat", "_ingested_at", "_source_system", "_batch_id")
          .withColumn("_batch_id", F.lit(BATCH)))

    upsert(df, "gold.fact_devoluciones", ["id_devolucion"])
    log_run("gold", "fact_devoluciones", df.count(), 0, time.time() - t0, batch=BATCH)
    print(f"  [DONE] gold.fact_devoluciones")


## fact_rfm_clientes
Necesidad: segmentar programa fidelización en 5 grupos de valor. Score sobre últimos 90 días, actualización semanal. Solo clientes activos (al menos 1 compra en 180 días). Linaje:
- R <- días desde última transacción (menor = mejor)
- F <- número de transacciones en 90 días
- M <- valor monetario total en 90 días
- score_r/f/m <- quintil 1-5 (ntile sobre clientes activos)
- segmento_rfm <- concatenación R{n}-F{n}-M{n}
- grupo_valor <- agrupación en 5 segmentos de negocio

In [0]:
def build_fact_rfm():
    print(" === Inicio fact_rfm ===")
    t0 = time.time()
    ventas = spark.table(f"{CATALOG}.silver.trans_ventas")

    activos = (ventas
               .filter(F.col("fec_trans") >= F.date_sub(TODAY, 180))
               .filter(F.col("id_miembro") != "ANONIMO")
               .select("id_miembro").distinct())

    rfm_base = (ventas
                .filter(F.col("fec_trans") >= F.date_sub(TODAY, 90))
                .filter(F.col("id_miembro") != "ANONIMO")
                .join(activos, on="id_miembro", how="inner")
                .groupBy("id_miembro")
                .agg(
                    F.datediff(TODAY, F.max("fec_trans")).alias("R"),
                    F.count("id_trans").alias("F"),
                    F.round(F.sum(
                        F.col("qty_vendida") * F.col("precio_unitario_venta")
                    ), 2).alias("M")
                ))

    df = (rfm_base
          # R invertido: menos días desde última compra = score más alto
          .withColumn("score_r", (6 - F.ntile(5).over(Window.orderBy(F.col("R").asc()))).cast("int"))
          .withColumn("score_f", F.ntile(5).over(Window.orderBy(F.col("F").asc())).cast("int"))
          .withColumn("score_m", F.ntile(5).over(Window.orderBy(F.col("M").asc())).cast("int"))
          .withColumn("segmento_rfm",
              F.concat_ws("-",
                  F.concat(F.lit("R"), F.col("score_r")),
                  F.concat(F.lit("F"), F.col("score_f")),
                  F.concat(F.lit("M"), F.col("score_m"))))
          # 5 grupos de valor basados en score total
          .withColumn("score_total", F.col("score_r") + F.col("score_f") + F.col("score_m"))
          .withColumn("grupo_valor",
              F.when(F.col("score_total") >= 13, F.lit("Champions"))
               .when(F.col("score_total") >= 10, F.lit("Leales"))
               .when(F.col("score_total") >= 7, F.lit("Potenciales"))
               .when(F.col("score_total") >= 4, F.lit("En Riesgo"))
               .otherwise(F.lit("Inactivos")))
          .withColumn("fec_calculo", TODAY)
          .withColumn("_batch_id",   F.lit(BATCH)))

    upsert(df, "gold.fact_rfm_clientes", ["id_miembro"])
    log_run("gold", "fact_rfm_clientes", df.count(), 0, time.time() - t0, batch=BATCH)
    print(f"  [DONE] gold.fact_rfm_clientes")


## agg_conversion_canal
- Necesidad: tasa de conversión y ticket promedio por canal y categoría
- Proxy de conversión: transacciones únicas / miembros únicos que compraron
  - (sin datos de visitas, la conversión se aproxima con clientes que compraron al menos una vez vs total de clientes activos por canal)

In [0]:
def build_agg_conversion_canal():
      print(" === Inicio agg_conversion_canal ===")
      t0 = time.time()
      ventas = spark.table(f"{CATALOG}.gold.fact_ventas")
      clientes = spark.table(f"{CATALOG}.gold.dim_clientes").select("id_miembro", "canal_pref")

      # Clientes activos por canal preferido (denominador de conversión)
      activos_canal = (clientes
                        .filter(F.col("id_miembro") != "ANONIMO")
                        .groupBy("canal_pref")
                        .agg(F.count("id_miembro").alias("clientes_activos_canal")))

      df = (ventas
            .filter(F.col("id_miembro") != "ANONIMO")
            .groupBy("canal_venta", "id_categ_n1")
            .agg(
                  F.countDistinct("id_miembro").alias("clientes_compraron"),
                  F.count("id_trans").alias("n_transacciones"),
                  F.round(F.sum("vr_venta_neto"), 2).alias("venta_neta_total"),
                  F.round(F.avg("vr_venta_neto"), 2).alias("ticket_promedio"),
                  F.round(F.avg("descuento_aplicado") * 100, 2).alias("tasa_descuento_prom_pct"),
            )
            .join(activos_canal, F.col("canal_venta") == F.col("canal_pref"), how="left")
            .withColumn("tasa_conversion_pct",
                  F.when(F.col("clientes_activos_canal") > 0,
                         F.round(F.col("clientes_compraron") / F.col("clientes_activos_canal") * 100, 2))
                  .otherwise(F.lit(None)))
            .drop("canal_pref")
            .withColumn("fec_calculo", TODAY)
            .withColumn("_batch_id", F.lit(BATCH)))

      upsert(df, "gold.agg_conversion_canal", ["canal_venta", "id_categ_n1", "fec_calculo"])
      log_run("gold", "agg_conversion_canal", df.count(), 0, time.time() - t0, batch=BATCH)
      print(f"  [DONE] gold.agg_conversion_canal")


## agg_ventas_diarias
- Necesidad: dashboard ejecutivo — ventas por país, tienda, canal y categoría con comparativo semana anterior

In [0]:
def build_agg_ventas_diarias():
    print(" === Inicio agg_ventas_diarias ===")
    t0 = time.time()
    fact = spark.table(f"{CATALOG}.gold.fact_ventas")

    # Agregación diaria base
    agg = (fact
           .groupBy("fec_trans", "id_pais", "id_tienda", "tipo_tienda", "canal_venta", "id_categ_n1")
           .agg(
               F.round(F.sum("vr_venta_neto"), 2).alias("venta_neta"),
               F.count("id_trans").alias("n_transacciones"),
               F.round(F.avg("vr_venta_neto"), 2).alias("ticket_promedio"),
               F.round(F.avg("descuento_aplicado") * 100, 2).alias("tasa_descuento_pct"),
               F.countDistinct("id_miembro").alias("clientes_unicos"),
           ))

    # Comparativo mismo día semana anterior via self-join
    agg_ant = (agg
               .withColumn("fec_trans", F.date_add(F.col("fec_trans"), 7))
               .withColumnRenamed("venta_neta", "venta_neta_sem_ant")
               .select("fec_trans", "id_pais", "id_tienda", "canal_venta", "id_categ_n1", "venta_neta_sem_ant"))

    df = (agg
          .join(agg_ant, on=["fec_trans", "id_pais", "id_tienda", "canal_venta", "id_categ_n1"], how="left")
          .withColumn("variacion_vs_sem_ant_pct",
              F.when(F.col("venta_neta_sem_ant") > 0,
                     F.round((F.col("venta_neta") - F.col("venta_neta_sem_ant")) / F.col("venta_neta_sem_ant") * 100, 2))
               .otherwise(F.lit(None)))
          .withColumn("_batch_id", F.lit(BATCH)))

    (df.write.format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .partitionBy("id_pais", "fec_trans")
       .saveAsTable(f"{CATALOG}.gold.agg_ventas_diarias"))
    log_run("gold", "agg_ventas_diarias", df.count(), 0, time.time() - t0, batch=BATCH)

    top10 = (fact
             .groupBy("fec_trans", "id_categ_n1", "art_id")
             .agg(
                 F.round(F.sum("vr_venta_neto"), 2).alias("venta_neta"),
                 F.count("id_trans").alias("n_transacciones"),)
             .withColumn("rank", F.row_number().over(Window.partitionBy("fec_trans", "id_categ_n1").orderBy(F.col("venta_neta").desc())))
             .filter(F.col("rank") <= 10)
             .drop("rank")
             .withColumn("_batch_id", F.lit(BATCH)))
    upsert(top10, "gold.agg_top10_articulos_diario", ["fec_trans", "id_categ_n1", "art_id"])

    print(f"  [DONE] gold.agg_ventas_diarias")


## Execution

In [0]:
print(f"=== Gold Facts | batch={BATCH} ===")
# fact_ventas primero — las demás agregaciones dependen de ella
try: build_fact_ventas()
except Exception as e: print(f"  [ERROR] fact_ventas: {e}")
try: build_fact_inventario()
except Exception as e: print(f"  [ERROR] fact_inventario: {e}")
try: build_fact_devoluciones()
except Exception as e: print(f"  [ERROR] fact_devoluciones: {e}")
try: build_fact_rfm()
except Exception as e: print(f"  [ERROR] fact_rfm_clientes: {e}")
try: build_agg_conversion_canal()
except Exception as e: print(f"  [ERROR] agg_conversion_canal: {e}")
try: build_agg_ventas_diarias()
except Exception as e: print(f"  [ERROR] agg_ventas_diarias: {e}")
print("\n[DONE] Gold facts completado.")
